# Stage II (lesions) - 03. Operating thresholds and test-set metrics

1. **Configuration** - trained checkpoints of the four lesion models.
2. **Validation threshold sweep** - pixel-pooled Dice, IoU, recall, precision and specificity
   on the validation split over each lesion's grid (MA 0.001-0.5 step 0.005; HE 0.20-0.70 step
   0.05; EX 0.20-0.65 step 0.025 with flip TTA; CWS 0.05-0.60 step 0.025 with flip TTA).
   HE (0.20) and EX (0.325) use the Dice optimum.
3. **Wide MA sweep** - the sweep quoted in the paper (0.001-0.009, 0.01-0.09, 0.10-0.50):
   Dice and recall keep rising as the threshold falls, so 0.01 was deployed for MA. For CWS the
   Dice optimum is 0.575 but 0.40 was deployed for higher recall. The test split plays no role.
4. **Test-set metrics** - at the deployed thresholds: pixel-pooled metrics and AUCs, image Dice
   (mean and SD), lesion-level detection (IoU >= 0.3; >= 0.2 for MA) and image-level
   sensitivity. EX and CWS average identity, horizontal- and vertical-flip predictions; CWS
   image Dice and lesion-level metrics use the closed and filtered masks.

## Step 1. Configuration

In [ ]:
from pathlib import Path

LESIONS = ["EX", "HE", "CWS", "MA"]  # order of the paper table
DATA_ROOT = Path("data/stage2_lesions")                 # manifests from notebook 01
CHECKPOINT_ROOT = Path("checkpoints/stage2/hsmoe_aunet")  # <LESION>/best_model.pth
RESULTS_DIR = Path("outputs/stage2_lesion_eval")
NUM_WORKERS = 0

In [ ]:
import gc
import json

import pandas as pd
import torch

from hierarchiretina.stage2.hsmoe_aunet import get_lesion_config, load_hsmoe_aunet
from hierarchiretina.stage2.lesion_data import build_dataloaders, load_splits
from hierarchiretina.stage2.lesion_metrics import (best_threshold, evaluate_test_set,
                                                   paper_table_row, sweep_thresholds_for,
                                                   threshold_sweep, wide_ma_thresholds)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def load(lesion):
    cfg = get_lesion_config(lesion)
    tr, va, te = load_splits(DATA_ROOT / lesion / "manifest.csv", lesion)
    _, val_loader, test_loader = build_dataloaders(lesion, tr, va, te, num_workers=NUM_WORKERS)
    ckpt = CHECKPOINT_ROOT / lesion / "best_model.pth"
    model = load_hsmoe_aunet(lesion, ckpt, map_location=DEVICE).to(DEVICE).eval()
    return cfg, model, val_loader, test_loader

## Step 2. Validation threshold sweep (each notebook's Cell 8.0)

In [ ]:
sweep_summary = []
for lesion in LESIONS:
    cfg, model, val_loader, _ = load(lesion)
    sweep = threshold_sweep(model, val_loader, sweep_thresholds_for(lesion), DEVICE,
                            tta=cfg.sweep_tta, use_amp=USE_AMP)
    sweep.to_csv(RESULTS_DIR / f"{lesion}_val_threshold_sweep.csv", index=False)
    t_opt = best_threshold(sweep)
    sweep_summary.append({"lesion": lesion, "Dice-optimal": t_opt,
                          "val Dice at optimum": sweep["dice"].max(),
                          "deployed": cfg.threshold, "note": cfg.threshold_note})
    del model
    gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(sweep_summary)

## Step 3. Wide MA validation sweep

Logits are cast to FP32 before the sigmoid (as in the sweep script used for the paper), and
the ratios use eps = 1e-9.

In [ ]:
cfg, model, val_loader, _ = load("MA")
wide = threshold_sweep(model, val_loader, wide_ma_thresholds(), DEVICE, tta=False,
                       use_amp=USE_AMP, fp32_sigmoid=True, eps=1e-9)
wide.to_csv(RESULTS_DIR / "MA_val_threshold_sweep_wide.csv", index=False)
del model
gc.collect(); torch.cuda.empty_cache()
wide[["threshold", "dice", "recall", "precision", "specificity"]].round(4)

## Step 4. Test-set metrics at the deployed thresholds (each notebook's Cell 8.5)

In [ ]:
rows = []
for lesion in LESIONS:
    cfg, model, _, test_loader = load(lesion)
    summary, per_image = evaluate_test_set(model, test_loader, lesion, DEVICE, use_amp=USE_AMP)
    (RESULTS_DIR / f"{lesion}_test_metrics_summary.json").write_text(json.dumps(summary, indent=2))
    per_image.to_csv(RESULTS_DIR / f"{lesion}_test_metrics_per_image.csv", index=False)
    rows.append(paper_table_row(summary))
    del model
    gc.collect(); torch.cuda.empty_cache()

table = pd.DataFrame(rows)
table.to_csv(RESULTS_DIR / "stage2_lesion_test_table.csv", index=False)
table.round(3)